# CryoSat-2 L1b 通用处理脚本 (SAR / SARIn)

基于 Tilling et al. (2018) 算法。
通过 `get_processing_params()` 自动识别文件模式，统一处理 SAR 和 SARIn 两种数据。

**主要差异对照表：**

| 参数 | SAR | SARIn |
|---|---|---|
| 原始 range bins | 256 | 1024 |
| 标称参考点 $b_n$ | 128 | 512 |
| 裁剪后统一 bins | 128 | 128 |
| SSD 阈值 | 6.29 | 4.62 |
| 首峰检测门槛 | 0.15 | 0.45 |
| 波形平滑 | 否 | 是（3-point MA）|


## 0. 模式自动识别 & 参数工厂

In [ ]:
import os

def get_processing_params(file_path):
    """
    根据 CryoSat-2 L1b 文件名自动判断模式，
    返回 Tilling et al. (2018) 算法所需的全部参数。
    """
    filename = os.path.basename(file_path)

    # ---- 1. 自动判定模式 ----
    if "SIR_SIN" in filename:
        mode = "SARIn"
    elif "SIR_SAR" in filename:
        mode = "SAR"
    else:
        raise ValueError(f"无法从文件名识别模式，请检查输入文件: {filename}")

    # ---- 2. 公共参数 ----
    params = {
        "mode"                 : mode,
        "c"                    : 299792458.0,   # 光速 (m/s)
        "bin_size"             : 0.2342,         # 每个 bin 对应的距离 (m)
        "cropped_bins"         : 128,            # 统一裁剪长度 (Tilling 2018)
        "crop_left"            : 50,             # 峰值左侧保留 bins
        "crop_right"           : 77,             # 峰值右侧保留 bins
        "retrack_threshold"    : 0.70,           # TFMRA 重跟踪阈值 (70%)
        "first_peak_min_ratio" : 0.20,           # 首峰最低占比阈值
        "pp_lead_threshold"    : 18,             # PP > 此值 -> Lead
        "pp_ice_threshold"     : 9,              # PP < 此值 -> Ice
    }

    # ---- 3. 模式专属参数 ----
    if mode == "SARIn":
        params["raw_bins"]          = 1024   # L1b 原始波形长度
        params["bn"]                = 512    # 标称参考点
        params["ssd_threshold"]     = 4.62   # SSD 分类阈值
        params["peak_threshold"]    = 0.45   # 首峰检测门槛
        params["needs_smoothing"]   = True   # 漫反射重跟踪前做 3-point MA
    elif mode == "SAR":
        params["raw_bins"]          = 256    # L1b 原始波形长度
        params["bn"]                = 128    # 标称参考点
        params["ssd_threshold"]     = 6.29   # SSD 分类阈值
        params["peak_threshold"]    = 0.15   # 首峰检测门槛
        params["needs_smoothing"]   = False  # SAR 波形相对干净

    return params


# ==========================================
# 路径配置（修改为你实际的文件路径）
# ==========================================
# L1_path    = r"E:\NWP\CS2_L1\2022\CS_OFFL_SIR_SIN_1B_20220323T224728_20220323T224947_E001.nc"
L1_path = r"C:\Users\TJ002\Desktop\CS_OFFL_SIR_SAR_1B_20250331T150910_20250331T151136_E001.nc"
L2_path    = r"E:\NWP\CS2_L2_within_region\2022\CS_OFFL_SIR_SIN_2__20220323T224728_20220323T224947_E001.nc"
region_shp = r"C:\Users\TJ002\Desktop\code\Cal_code_data\NWP_orbit_processing\Arctic_Canada_North.shp"
mss_file   = r"E:\Project_2024\CryoSat-2 L1\DTU21MSS_1min_WGS84.nc"
snow_dir   = r"D:\S1_CS2_data\awi_snow_merged"

# 一键获取当前文件的全部参数
P = get_processing_params(L1_path)

# 把常用标量解包到顶层，方便后续代码直接使用
mode       = P["mode"]
c          = P["c"]
bin_size   = P["bin_size"]
bn         = P["bn"]

print(f"当前处理模式  : {mode}")
print(f"原始 range bins : {P['raw_bins']}")
print(f"标称参考点 bn  : {bn}")
print(f"SSD 阈值       : {P['ssd_threshold']}")
print(f"首峰检测门槛   : {P['peak_threshold']}")
print(f"需要平滑       : {P['needs_smoothing']}")


## 1. 研究区域加载

In [10]:
import xarray as xr
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
from scipy.interpolate import griddata

# # 读取研究区域的 Shapefile
# region_gdf = gpd.read_file(region_shp)
# if region_gdf.crs is None or region_gdf.crs.to_epsg() != 4326:
#     region_gdf = region_gdf.to_crs(epsg=4326)
# 经度覆盖全球：-180 到 180
min_lon, max_lon = -180.0, 180.0

# 纬度从北极圈 (约 66.56°N) 到北极点 (90°N)
# 注：有些研究也会宽泛地使用 60°N 作为北极研究的南界，你可以根据需要将 66.56 改为 60.0
min_lat, max_lat = 66.56, 90.0
# min_lon, min_lat, max_lon, max_lat = region_gdf.total_bounds
print(f"Region boundary: lon({min_lon:.2f}, {max_lon:.2f}), lat({min_lat:.2f}, {max_lat:.2f})")


Region boundary: lon(-180.00, 180.00), lat(66.56, 90.00)


## 2. 读取 L1b 数据

In [11]:
ds = xr.open_dataset(L1_path)

time_20_ku          = ds['time_20_ku'].values
lat_20_ku           = ds['lat_20_ku'].values
lon_20_ku           = ds['lon_20_ku'].values
waveform_20_ku      = ds['pwr_waveform_20_ku'].values
noise_power_20_ku   = ds['noise_power_20_ku'].values
noise_power_real    = np.where(noise_power_20_ku == -9999.99, np.nan, noise_power_20_ku)
stack_std_20_ku     = ds['stack_std_20_ku'].values
stack_peakiness_20_ku = ds['stack_peakiness_20_ku'].values
stack_kurtosis_20_ku  = ds['stack_kurtosis_20_ku'].values
window_del_20_ku    = ds['window_del_20_ku'].values
alt                 = ds['alt_20_ku'].values

# ---- 核实 raw_bins 与文件一致 ----
n_bins_detected = waveform_20_ku.shape[1]
if n_bins_detected != P['raw_bins']:
    print(f"⚠️  警告：文件实际 bins={n_bins_detected}，参数中 raw_bins={P['raw_bins']}，请核查文件/模式！")
else:
    print(f"✅  bins 校验通过：{n_bins_detected} bins ({mode} 模式）")

# ---- 时间转换 ----
tai_epoch   = datetime(2000, 1, 1, 0, 0, 0)
tai_seconds = (time_20_ku - np.datetime64('2000-01-01T00:00:00')) / np.timedelta64(1, 's')
utc_time    = np.array([tai_epoch + timedelta(seconds=t) for t in tai_seconds])

# ---- 扩展 surf_type 到 20Hz ----
surf_type        = ds["surf_type_01"].values
ind_first_meas   = ds["ind_first_meas_20hz_01"].values.astype(int)
N_20             = lat_20_ku.shape[0]
surf_type_20hz   = np.full(N_20, np.nan)
for i in range(len(ind_first_meas)):
    start_idx = ind_first_meas[i]
    end_idx   = min(start_idx + 20, N_20)
    surf_type_20hz[start_idx:end_idx] = surf_type[i]

# ---- 构建 DataFrame ----
df = pd.DataFrame({
    'time_20_ku'       : time_20_ku,
    'utc_time'         : utc_time,
    'alt'              : alt,
    'noise_power_real' : noise_power_real,
    'std'              : stack_std_20_ku,
    'pp'               : stack_peakiness_20_ku,
    'k'                : stack_kurtosis_20_ku,
    'lat'              : lat_20_ku,
    'lon'              : lon_20_ku,
    'surf_type'        : surf_type_20hz,
    'window_del'       : window_del_20_ku,
})
df['window_del_seconds'] = df['window_del'].dt.total_seconds()

print(f"读取完成：{len(df)} 个 20Hz 测量点")


✅  bins 校验通过：256 bins (SAR 模式）
读取完成：3224 个 20Hz 测量点


## 3. 波形分类（PP + SSD，Tilling 2018）

裁剪方法：找到最大功率所在 bin ($b_{max}$)，提取前 50 个 + 后 77 个 bins，统一为 128 bins。

注意：`bn`（标称参考点）**仅用于重跟踪距离校正**，不影响裁剪逻辑。

In [ ]:
import scipy.signal

def crop_waveform_and_get_offset(waveform, crop_left=50, crop_right=77):
    """
    将任意长度的波形裁剪为 (crop_left + crop_right + 1) = 128 bins。
    返回裁剪后波形和绝对偏移量 abs_offset = b_max - crop_left。
    适用于 SAR (256 bins) 和 SARIn (1024 bins)。
    """
    b_max       = np.argmax(waveform)
    start_index = b_max - crop_left
    end_index   = b_max + crop_right + 1

    pad_left  = max(0, -start_index)
    pad_right = max(0, end_index - len(waveform))
    start_safe = max(0, start_index)
    end_safe   = min(len(waveform), end_index)

    cropped = waveform[start_safe:end_safe]
    if pad_left > 0 or pad_right > 0:
        cropped = np.pad(cropped, (pad_left, pad_right), 'constant', constant_values=0)

    abs_offset = b_max - crop_left  # 裁剪窗口左边缘在原始数组中的绝对位置
    return cropped, abs_offset


def process_waveform_tilling_2018(waveform_array, ssd_array, params):
    """
    通用版波形分类函数。
    输入 params 字典即可自动适配 SAR / SARIn。
    返回：types, custom_pps, absolute_offsets, cropped_waveforms
    """
    n_samples      = len(waveform_array)
    ssd_threshold  = params['ssd_threshold']
    pp_lead        = params['pp_lead_threshold']
    pp_ice         = params['pp_ice_threshold']
    crop_left      = params['crop_left']
    crop_right     = params['crop_right']

    types            = np.full(n_samples, 'unknown', dtype=object)
    absolute_offsets = np.full(n_samples, np.nan)
    custom_pps       = np.full(n_samples, np.nan)
    cropped_waveforms = []

    for i in range(n_samples):
        wf  = waveform_array[i]
        ssd = ssd_array[i]

        # --- 1. 裁剪波形到 128 bins ---
        cropped_wf, abs_offset = crop_waveform_and_get_offset(wf, crop_left, crop_right)
        cropped_waveforms.append(cropped_wf)
        absolute_offsets[i] = abs_offset

        # --- 2. 计算 Pulse Peakiness (PP) ---
        # 噪声底板：裁剪后波形 bin 10~19 的均值
        noise_floor = np.mean(cropped_wf[10:20])
        valid_bins  = cropped_wf[cropped_wf > noise_floor]
        if len(valid_bins) > 0:
            p_mean = np.mean(valid_bins)
            p_max  = np.max(cropped_wf)
            pp     = p_max / p_mean if p_mean > 0 else 0
        else:
            pp = 0
        custom_pps[i] = pp

        # --- 3. 分类 ---
        if pp > pp_lead and ssd < ssd_threshold:
            types[i] = 'lead'
        elif pp < pp_ice and ssd > ssd_threshold:
            types[i] = 'ice'

    return types, custom_pps, absolute_offsets, np.array(cropped_waveforms)


# ---- 执行分类 ----
classified_types, calculated_pps, abs_offsets, cropped_wfs = process_waveform_tilling_2018(
    waveform_20_ku, df['std'].values, P
)
df['type']            = classified_types
df['tilling_pp']      = calculated_pps
df['abs_offset']      = abs_offsets
df['cropped_waveform'] = list(cropped_wfs)

print(f"[{mode} 模式] Lead 点数: {np.sum(df['type'] == 'lead')}")
print(f"[{mode} 模式] Ice  点数: {np.sum(df['type'] == 'ice')}")


## 4. 重跟踪（Retracking）

- **Lead（镜面回波）**：Giles 函数拟合（Levenberg-Marquardt）
- **Ice（漫反射回波）**：TFMRA 70% 阈值前缘追踪

关键公式：$C_R = (b_0 - b_n) \times \delta_b$

其中 $b_n$ 为模式专属标称参考点（SAR=128，SARIn=512）。

In [13]:
import scipy.ndimage
from scipy.optimize import curve_fit


# ============================================================
# 漫反射重跟踪：TFMRA 70% 阈值法
# ============================================================
def retrack_diffuse_waveform_tilling(waveform_array, params):
    """
    通用漫反射重跟踪（Ice floe）。
    - 若 params['needs_smoothing'] 为 True，先做 3-point 移动平均（SARIn 模式）。
    - bn 从 params 中读取，不再靠 total_bins//2 推算。
    """
    n_samples         = len(waveform_array)
    range_correction  = np.full(n_samples, np.nan)
    b_n               = params['bn']               # 标称参考点
    bin_size          = params['bin_size']          # 0.2342 m/bin
    needs_smoothing   = params['needs_smoothing']
    crop_left         = params['crop_left']
    crop_right        = params['crop_right']
    peak_min_ratio    = params['first_peak_min_ratio']
    retrack_thr       = params['retrack_threshold']  # 0.70

    for i in range(n_samples):
        raw_wf = waveform_array[i]

        # 1. 裁剪
        cropped_wf, abs_offset = crop_waveform_and_get_offset(raw_wf, crop_left, crop_right)

        # 2. 可选平滑（SARIn 必须，SAR 可选）
        if needs_smoothing:
            wf_proc = scipy.ndimage.uniform_filter1d(cropped_wf, size=3)
        else:
            wf_proc = cropped_wf.copy()

        # 3. 寻找首个有效峰值（>= 20% 全局最大）
        max_val = np.max(wf_proc)
        if max_val <= 0:
            continue
        peaks, _ = scipy.signal.find_peaks(wf_proc)
        valid_peaks = [p for p in peaks if wf_proc[p] >= peak_min_ratio * max_val]
        if not valid_peaks:
            continue

        first_peak_idx = valid_peaks[0]
        first_peak_val = wf_proc[first_peak_idx]

        # 4. 定位 70% 阈值：从峰值向左回溯
        threshold_70 = retrack_thr * first_peak_val
        b_0_local = np.nan
        for j in range(first_peak_idx - 1, -1, -1):
            if wf_proc[j] <= threshold_70 <= wf_proc[j + 1]:
                frac      = (threshold_70 - wf_proc[j]) / (wf_proc[j + 1] - wf_proc[j])
                b_0_local = j + frac
                break
        if np.isnan(b_0_local):
            continue

        # 5. 局部 -> 全局绝对 bin 编号
        b_0_global = abs_offset + b_0_local

        # 6. 距离校正 C_R = (b0 - bn) * delta_b
        range_correction[i] = (b_0_global - b_n) * bin_size

    return range_correction


# ============================================================
# 镜面重跟踪：Giles 函数拟合
# ============================================================
def giles_echo_function(t, a, t0, k, sigma):
    """Giles 分段镜面回波模型（Tilling 2018）。"""
    sigma = max(sigma, 1e-6)
    k     = max(k, 1e-6)
    tb    = k * (sigma ** 2)
    sqrt_k_tb = np.sqrt(k * tb)
    a2 = (5 * k * sigma - 4 * sqrt_k_tb) / (2 * sigma * tb * sqrt_k_tb)
    a3 = (2 * sqrt_k_tb - 3 * k * sigma) / (2 * sigma * (tb ** 2) * sqrt_k_tb)

    f = np.zeros_like(t, dtype=float)
    mask1 = t < t0
    f[mask1] = (t[mask1] - t0) / sigma
    mask2 = (t >= t0) & (t < (tb + t0))
    dt2   = t[mask2] - t0
    f[mask2] = a3 * (dt2 ** 3) + a2 * (dt2 ** 2) + (1 / sigma) * dt2
    mask3 = t >= (tb + t0)
    dt3   = np.maximum(t[mask3] - t0, 0)
    f[mask3] = np.sqrt(k * dt3)

    return a * np.exp(-(f ** 2))


def retrack_specular_waveform_giles(waveform_array, params):
    """
    通用镜面重跟踪（Lead）。
    Giles 函数在裁剪后的 128-bin 空间内拟合，
    随后将局部 t0 映射回原始绝对编号后计算 C_R。
    """
    n_samples        = len(waveform_array)
    range_correction = np.full(n_samples, np.nan)
    b_n              = params['bn']
    bin_size         = params['bin_size']
    crop_left        = params['crop_left']
    crop_right       = params['crop_right']

    t_128 = np.arange(128, dtype=float)  # 裁剪后的相对 x 轴

    for i in range(n_samples):
        raw_wf = waveform_array[i]
        cropped_wf, abs_offset = crop_waveform_and_get_offset(raw_wf, crop_left, crop_right)

        a_guess     = np.max(cropped_wf)
        t0_guess    = float(np.argmax(cropped_wf))
        p0          = [a_guess, t0_guess, 0.5, 1.0]

        try:
            popt, _ = curve_fit(
                giles_echo_function,
                t_128, cropped_wf,
                p0=p0, method='lm', maxfev=3000
            )
            t0_local = popt[1]
            if not (0 <= t0_local <= 127):
                continue
        except RuntimeError:
            continue

        b_0_global = abs_offset + t0_local
        range_correction[i] = (b_0_global - b_n) * bin_size

    return range_correction


print("重跟踪函数定义完成。")


重跟踪函数定义完成。


## 5. 区域裁剪 + 重跟踪执行 + 高程计算

In [ ]:
# ---- 噪声去除 ----
noise_floor_arr = np.nan_to_num(noise_power_real, nan=0.0)
waveform        = waveform_20_ku - noise_floor_arr[:, np.newaxis]
window_del      = df['window_del_seconds'].values

# ---- 区域裁剪（使用 shp 边界） ----
region_mask = (
    (df['lat'] >= min_lat) & (df['lat'] <= max_lat) &
    (df['lon'] >= min_lon) & (df['lon'] <= max_lon)
)
df_region         = df[region_mask].copy()
waveform_region   = waveform[region_mask]
window_del_region = window_del[region_mask]

# ---- 初始化列 ----
df_region['range_correction'] = np.nan
df_region['range_final']      = np.nan
df_region['range_window_del'] = window_del_region * c / 2

# ---- Lead & Ice mask（仅海洋区域，surf_type 0 或 1）----
ocean_surf   = (df_region['surf_type'] == 0) | (df_region['surf_type'] == 1)
lead_mask_r  = (df_region['type'] == 'lead') & ocean_surf
ice_mask_r   = (df_region['type'] == 'ice')  & ocean_surf

# ---- Lead 重跟踪（Giles 拟合）----
valid_wf_lead = waveform_region[lead_mask_r]
if len(valid_wf_lead) > 0:
    print(f"正在对 {len(valid_wf_lead)} 个 Lead 点做 Giles 拟合...")
    TFMRA_lead = retrack_specular_waveform_giles(valid_wf_lead, P)
    df_region.loc[lead_mask_r, 'range_correction'] = TFMRA_lead
    print("Lead 重跟踪完成。")

# ---- Ice 重跟踪（TFMRA 70%）----
valid_wf_ice = waveform_region[ice_mask_r]
if len(valid_wf_ice) > 0:
    print(f"正在对 {len(valid_wf_ice)} 个 Ice 点做 TFMRA 重跟踪...")
    TFMRA_ice = retrack_diffuse_waveform_tilling(valid_wf_ice, P)
    # 漫反射 vs 镜面固定偏差校正：-16.26 cm (Tilling 2018)
    df_region.loc[ice_mask_r, 'range_correction'] = TFMRA_ice - 0.1626
    print("Ice 重跟踪完成。")

# ---- 最终高程 ----
valid_corr = df_region['range_correction'].notna()
df_region.loc[valid_corr, 'range_final'] = (
    df_region.loc[valid_corr, 'range_window_del'] +
    df_region.loc[valid_corr, 'range_correction']
)
df_region['elevation'] = df_region['alt'] - df_region['range_final']

print(f"\n高程计算完成，有效点数: {df_region['elevation'].notna().sum()}")


In [ ]:
# 可视化
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(df_region['range_correction'].dropna(), marker='.', linestyle='none', markersize=2)
plt.title(f'Range Correction (m) [{mode}]')
plt.ylabel('Correction (m)')

plt.subplot(1, 2, 2)
plt.plot(df_region['elevation'].dropna(), marker='.', linestyle='none', markersize=2)
plt.title(f'Surface Elevation (m) [{mode}]')
plt.ylabel('Elevation (m)')
plt.tight_layout()
plt.show()


## 6. 地球物理校正（Geophysical Corrections）

In [ ]:
correction_vars = [
    'mod_dry_tropo_cor_01', 'mod_wet_tropo_cor_01', 'inv_bar_cor_01',
    'iono_cor_01',
    'ocean_tide_01', 'ocean_tide_eq_01', 'load_tide_01',
    'solid_earth_tide_01', 'pole_tide_01'
]

N_20 = len(time_20_ku)
corrections_20hz = {var: np.full(N_20, np.nan) for var in correction_vars}

for i in range(len(ind_first_meas)):
    start_idx = ind_first_meas[i]
    end_idx   = min(start_idx + 20, N_20)
    for var in correction_vars:
        corrections_20hz[var][start_idx:end_idx] = ds[var].values[i]

df_corrections_20hz = pd.DataFrame(corrections_20hz)
df_corrections_20hz["time_20_ku"] = time_20_ku

corrections_sum = sum(df_corrections_20hz[var] for var in correction_vars)
df['corrections_sum'] = corrections_sum

print("地球物理校正扩展完成。")
print(df_corrections_20hz[correction_vars].describe())


In [ ]:
# 写入 df_region 并计算校正后高程
df_L1 = df_region.copy()
df_L1['C_G'] = df['corrections_sum']
df_L1['R0_True'] = df_L1['range_final'] + df_L1['C_G']
df_L1['elevation_corrected'] = df_L1['alt'] - df_L1['R0_True']

plt.plot(df_L1['elevation_corrected'].dropna())
plt.title(f'Corrected Surface Elevation [{mode}]')
plt.ylabel('Elevation (m)')
plt.grid(True)
plt.show()


## 7. MSS 插值

In [18]:
ds_mss = xr.open_dataset(mss_file)
lon_mss = ds_mss['lon'].values
lon_mss[lon_mss > 180] -= 360

min_lon_l1, max_lon_l1 = df_L1['lon'].min(), df_L1['lon'].max()
min_lat_l1, max_lat_l1 = df_L1['lat'].min(), df_L1['lat'].max()
print(f"自动识别裁剪范围: lon({min_lon_l1:.2f}, {max_lon_l1:.2f}), lat({min_lat_l1:.2f}, {max_lat_l1:.2f})")

mss_clipped = ds_mss.where(
    (ds_mss.lon >= min_lon_l1) & (ds_mss.lon <= max_lon_l1) &
    (ds_mss.lat >= min_lat_l1) & (ds_mss.lat <= max_lat_l1),
    drop=True
)
lon_grid, lat_grid = np.meshgrid(mss_clipped.lon.values, mss_clipped.lat.values)

mss_interp = griddata(
    (lon_grid.flatten(), lat_grid.flatten()),
    mss_clipped.mean_sea_surf_sol2.values.flatten(),
    (df_L1['lon'], df_L1['lat']),
    method='linear'
)
df_L1['mss_interp'] = mss_interp
print("MSS 插值完成。")


自动识别裁剪范围: lon(-48.33, 78.01), lat(83.66, 88.00)
MSS 插值完成。


## 8. 地表高程 (h_surface) 计算

In [ ]:
df_L1['h_surface'] = df_L1['alt'] - df_L1['range_final'] - df_L1['C_G']

plt.figure(figsize=(10, 5))
sc = plt.scatter(df_L1['lon'], df_L1['lat'], c=df_L1['h_surface'], cmap='viridis', s=10)
plt.colorbar(sc, label='Surface Elevation (m)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.title(f'Surface Elevation [{mode}]')
plt.show()


## 9. SLA 计算（Sea Level Anomaly）

In [20]:
from scipy.interpolate import interp1d
from scipy.ndimage import uniform_filter1d

lead_mask = df_L1['type'] == 'lead'

# SLA_raw（仅 lead 点）
df_L1['SLA_raw'] = np.nan
df_L1.loc[lead_mask, 'SLA_raw'] = (
    df_L1.loc[lead_mask, 'h_surface'] - df_L1.loc[lead_mask, 'mss_interp']
)

# ±3m 异常值剔除
valid_lead_mask = lead_mask & (df_L1['SLA_raw'] >= -3.0) & (df_L1['SLA_raw'] <= 3.0)
valid_lead_indices  = df_L1.index[valid_lead_mask].to_numpy()
valid_SLA_raw_values = df_L1.loc[valid_lead_mask, 'SLA_raw'].to_numpy()
print(f"有效 Lead 点数（剔除 ±3m 后）: {len(valid_SLA_raw_values)}")

if len(valid_SLA_raw_values) > 1:
    # Step 1: 100 km 滑动平均
    window_100km = int(100000 / 380)
    SLA_raw_series    = pd.Series(valid_SLA_raw_values, index=valid_lead_indices)
    SLA_smoothed_raw  = SLA_raw_series.rolling(window=window_100km, center=True, min_periods=1).mean()

    # Step 2: 线性插值到全轨迹（不外推）
    interp_func = interp1d(
        SLA_smoothed_raw.index, SLA_smoothed_raw.values,
        kind='linear', bounds_error=False, fill_value=np.nan
    )
    df_L1['SLA_interp'] = interp_func(df_L1.index)
    df_L1['SLA_final']  = df_L1['SLA_interp']
    df_L1['SLA']        = df_L1['SLA_final']
else:
    print("警告：有效 lead 点不足，无法插值！")
    df_L1['SLA'] = np.nan

print(f"成功插值点数: {df_L1['SLA'].notna().sum()}")


有效 Lead 点数（剔除 ±3m 后）: 324
成功插值点数: 3084


## 10. 距离计算 + 200km 掩膜 + 雷达干舷

In [ ]:
from scipy.spatial import cKDTree
from geopy.distance import geodesic

# ---- 到最近 Lead 的距离（用于 200km 掩膜）----
R = 6371000.0
all_coords_rad  = np.radians(df_L1[['lat', 'lon']].to_numpy())
lead_coords_rad = np.radians(df_L1.loc[lead_mask, ['lat', 'lon']].to_numpy())

if len(lead_coords_rad) > 0:
    tree = cKDTree(lead_coords_rad)
    dist_rad, _ = tree.query(all_coords_rad, k=1)
    df_L1['dist_to_lead'] = dist_rad * R
else:
    df_L1['dist_to_lead'] = np.inf

# 超过 200km 的点 SLA 设为 NaN
df_L1.loc[df_L1['dist_to_lead'] > 200000, 'SLA'] = np.nan
print(f"200km 掩膜后有效 SLA 点数: {df_L1['SLA'].notna().sum()}")

# ---- 沿轨迹距离 ----
distance_km = [0]
for i in range(1, len(df_L1)):
    c1 = (df_L1['lat'].iloc[i-1], df_L1['lon'].iloc[i-1])
    c2 = (df_L1['lat'].iloc[i],   df_L1['lon'].iloc[i])
    distance_km.append(distance_km[-1] + geodesic(c1, c2).km)
df_L1['distance'] = distance_km

# ---- 雷达干舷（Radar Freeboard）----
ice_mask_l1 = df_L1['type'] == 'ice'
df_L1['radar_freeboard'] = np.nan
df_L1.loc[ice_mask_l1, 'radar_freeboard'] = (
    df_L1.loc[ice_mask_l1, 'h_surface'] -
    (df_L1.loc[ice_mask_l1, 'mss_interp'] + df_L1.loc[ice_mask_l1, 'SLA'])
)

# Tilling 2018：过滤范围 -0.25m ~ 2.25m（仅雷达干舷，校正前）
valid_fb = (df_L1['radar_freeboard'] >= -0.25) & (df_L1['radar_freeboard'] <= 2.25)
df_L1.loc[~valid_fb, 'radar_freeboard'] = np.nan

print(df_L1['radar_freeboard'].describe())


## 11. 雪深插值 & 校正干舷 & 海冰厚度

In [ ]:
import os

df_L1['utc_time'] = pd.to_datetime(df_L1['utc_time'])
df_L1['month']    = df_L1['utc_time'].dt.month
df_L1['f_myi']    = 0.0   # 默认全为一年冰（FYI）
df_L1['merged_sd_raw'] = np.nan
df_L1['w99_weight']    = np.nan
df_L1['hs']            = np.nan

for month in df_L1['month'].dropna().unique():
    snow_file = os.path.join(
        snow_dir,
        f"awi-siral-l4-snow_on_seaice-monthly_warren_amsr2_clim-{int(month):02d}-fv1p0.nc"
    )
    if not os.path.exists(snow_file):
        print(f"⚠️  找不到文件 {snow_file}")
        continue

    ds_snow   = xr.open_dataset(snow_file)
    lon_snow  = ds_snow['lon'].values
    lat_snow  = ds_snow['lat'].values
    lon_grid_s, lat_grid_s = np.meshgrid(lon_snow, lat_snow)
    snow_grid = ds_snow['merged_snow_depth'].values
    w_name    = 'w99_weight'
    weight_grid = ds_snow[w_name].values if w_name in ds_snow.data_vars else np.ones_like(snow_grid)

    lon_grid_180 = ((lon_grid_s + 180) % 360) - 180
    valid_mask_s = ~np.isnan(snow_grid)
    pts_valid    = (lon_grid_180[valid_mask_s], lat_grid_s[valid_mask_s])
    snow_valid   = snow_grid[valid_mask_s]
    weight_valid = weight_grid[valid_mask_s]

    month_mask = df_L1['month'] == month
    pts_lon_180 = ((df_L1.loc[month_mask, 'lon'].values + 180) % 360) - 180
    pts_lat     = df_L1.loc[month_mask, 'lat'].values

    interp_sd = griddata(pts_valid, snow_valid,   (pts_lon_180, pts_lat), method='linear')
    interp_w  = griddata(pts_valid, weight_valid, (pts_lon_180, pts_lat), method='linear')
    nan_mask  = np.isnan(interp_sd)
    if nan_mask.any():
        interp_sd[nan_mask] = griddata(pts_valid, snow_valid,   (pts_lon_180[nan_mask], pts_lat[nan_mask]), method='nearest')
        interp_w[nan_mask]  = griddata(pts_valid, weight_valid, (pts_lon_180[nan_mask], pts_lat[nan_mask]), method='nearest')

    df_L1.loc[month_mask, 'merged_sd_raw'] = interp_sd
    df_L1.loc[month_mask, 'w99_weight']    = interp_w
    print(f"✅  {int(month):02d} 月雪深插值完成。")

# 雪深修正（一年冰减半）
c_fyi           = 0.5
df_L1['c_factor'] = (1 - df_L1['f_myi']) * c_fyi * df_L1['w99_weight']
df_L1['hs']       = df_L1['merged_sd_raw'] - df_L1['c_factor'] * df_L1['merged_sd_raw']

# 校正干舷（Corrected Freeboard）
df_L1['corrected_freeboard'] = df_L1['radar_freeboard'] + 0.25 * df_L1['hs']

# Tilling 2018 最终过滤：-0.3m ~ 3.0m
valid_fc = (df_L1['corrected_freeboard'] >= -0.3) & (df_L1['corrected_freeboard'] <= 3.0)
df_L1.loc[~valid_fc, 'corrected_freeboard'] = np.nan

print(df_L1['corrected_freeboard'].describe())


In [ ]:
# ---- 海冰厚度（流体静力学平衡）----
rho_w = 1023.9
rho_s = 324.0
df_L1['rho_i'] = df_L1['f_myi'] * 882.0 + (1 - df_L1['f_myi']) * 916.7

df_L1['sea_ice_thickness'] = (
    (df_L1['corrected_freeboard'] * rho_w + df_L1['hs'] * rho_s) /
    (rho_w - df_L1['rho_i'])
)

print(f"\n🎉 [{mode}] 海冰厚度统计：")
print(df_L1['sea_ice_thickness'].dropna().describe())


## 12. 读取 L2 数据对比

In [24]:
from scipy.spatial import cKDTree

ds_L2 = xr.open_dataset(L2_path)

time_20_ku_L2       = ds_L2['time_20_ku'].values
height_1_20_ku      = ds_L2['height_1_20_ku'].values
mss_seaIce          = ds_L2['mean_sea_surf_sea_ice_01'].values
radar_freeboard_L2  = ds_L2['radar_freeboard_20_ku'].values
ssha                = ds_L2['ssha_interp_20_ku'].values
lon_L2              = ds_L2['lon_poca_20_ku'].values
lat_L2              = ds_L2['lat_poca_20_ku'].values
lat_01              = ds_L2['lat_01'].values
lon_01              = ds_L2['lon_01'].values
range_1             = ds_L2['range_1_20_ku'].values
snow_density        = ds_L2['snow_density_01'].values
snow_depth          = ds_L2['snow_depth_01'].values
alt_L2              = ds_L2['alt_01'].values
height_sea_ice_floe = ds_L2['height_sea_ice_floe_20_ku'].values
height_sea_ice_lead = ds_L2['height_sea_ice_lead_20_ku'].values
surf_type_L2        = ds_L2['surf_type_20_ku'].values

tai_s_L2 = (time_20_ku_L2 - np.datetime64('2000-01-01T00:00:00')) / np.timedelta64(1, 's')
utc_L2   = np.array([tai_epoch + timedelta(seconds=t) for t in tai_s_L2])


def match_nearest_valid_value(var_data, var_lat, var_lon, target_lat, target_lon, fill_value):
    """将 1Hz 变量最近邻匹配到 20Hz 目标点。"""
    valid_mask = (var_data != fill_value)
    tree       = cKDTree(np.c_[var_lat[valid_mask], var_lon[valid_mask]])
    _, indices = tree.query(np.c_[target_lat, target_lon], k=1)
    return var_data[valid_mask][indices]


fv_sss = ds_L2["mean_sea_surf_sea_ice_01"].attrs.get("_FillValue", np.nan)
fv_sd  = ds_L2["snow_density_01"].attrs.get("_FillValue", np.nan)
fv_sdp = ds_L2["snow_depth_01"].attrs.get("_FillValue", np.nan)

matched_mss    = match_nearest_valid_value(mss_seaIce,   lat_01, lon_01, lat_L2, lon_L2, fv_sss)
matched_sd     = match_nearest_valid_value(snow_density, lat_01, lon_01, lat_L2, lon_L2, fv_sd)
matched_sdp    = match_nearest_valid_value(snow_depth,   lat_01, lon_01, lat_L2, lon_L2, fv_sdp)
matched_alt_L2 = match_nearest_valid_value(alt_L2,       lat_01, lon_01, lat_L2, lon_L2, fv_sd)

df_L2 = pd.DataFrame({
    'time_20_ku'         : time_20_ku_L2,
    'utc_time'           : utc_L2,
    'height_1_20_ku'     : height_1_20_ku,
    'mss_seaIce'         : matched_mss,
    'radar_freeboard'    : radar_freeboard_L2,
    'ssha'               : ssha,
    'lat'                : lat_L2,
    'lon'                : lon_L2,
    'range_1'            : range_1,
    'snow_density'       : matched_sd,
    'snow_depth'         : matched_sdp,
    'alt'                : matched_alt_L2,
    'height_sea_ice_floe': height_sea_ice_floe,
    'height_sea_ice_lead': height_sea_ice_lead,
    'surf_type'          : surf_type_L2,
})

print(f"L2 读取完成：{len(df_L2)} 个点")


L2 读取完成：3084 个点


In [17]:
# L2 地球物理校正
corr_vars_L2 = [
    'mod_dry_tropo_cor_01', 'mod_wet_tropo_cor_01', 'inv_bar_cor_01',
    'iono_cor_01',
    'ocean_tide_01', 'ocean_tide_eq_01', 'load_tide_01',
    'solid_earth_tide_01', 'pole_tide_01'
]
for var in corr_vars_L2:
    data     = ds_L2[var].values
    fv       = ds_L2[var].attrs.get('_FillValue', np.nan)
    df_L2[var] = match_nearest_valid_value(data, lat_01, lon_01, lat_L2, lon_L2, fv)

df_L2['corrections_sum'] = df_L2[corr_vars_L2].sum(axis=1)

# L2 区域掩膜
valid_L2 = (
    ((df_L2['surf_type'] == 0) | (df_L2['surf_type'] == 1)) &
    (df_L2['lat'] >= df_L1['lat'].min()) & (df_L2['lat'] <= df_L1['lat'].max()) &
    (df_L2['lon'] >= df_L1['lon'].min()) & (df_L2['lon'] <= df_L1['lon'].max())
)
cols_to_mask = ['height_1_20_ku', 'mss_seaIce', 'radar_freeboard', 'ssha',
                'range_1', 'snow_density', 'snow_depth', 'alt',
                'height_sea_ice_floe', 'height_sea_ice_lead']
df_L2.loc[~valid_L2, cols_to_mask] = np.nan

# L2 沿轨迹距离
dist_L2 = [0]
for i in range(1, len(df_L2)):
    c1 = (df_L2['lat'].iloc[i-1], df_L2['lon'].iloc[i-1])
    c2 = (df_L2['lat'].iloc[i],   df_L2['lon'].iloc[i])
    dist_L2.append(dist_L2[-1] + geodesic(c1, c2).km)
df_L2['distance'] = dist_L2

# L2 计算干舷
df_L2['radar_freeboard_cal'] = df_L2['height_sea_ice_floe'] - (df_L2['mss_seaIce'] + df_L2['ssha'])
print("L2 处理完成。")


L2 处理完成。


## 13. 对比可视化

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(df_L1['distance'], df_L1['radar_freeboard'],     '.', label=f'L1 Radar Freeboard [{mode}]', alpha=0.6)
plt.plot(df_L2['distance'], df_L2['radar_freeboard_cal'], '.', label='L2 Calc Freeboard', alpha=0.6)
plt.xlabel('Distance (km)')
plt.ylabel('Freeboard (m)')
plt.legend()
plt.title(f'Comparison: L1 [{mode}] vs L2 Freeboard')
plt.grid(True)
plt.show()


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))

# 原始点：mss_seaIce
# plt.plot(df_L2['distance'], df_L2['ssha'], '.', color='royalblue', markersize=2, label='ssha_L2')

# 原始点：mss_interp
plt.plot(df_L1['distance'], df_L1['sea_ice_thickness'], '.', color='lightcoral', markersize=2, label='sea_ice_thickness')


# 图形设置
plt.xlabel('Distance / km', fontsize=12)
plt.ylabel('alt / m', fontsize=12)
plt.title('sea ice thickness', fontsize=14)
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# 假设你已经有这些变量：
# - df_L1（包含最终插值得到的 df_L1['SLA']）
# - SLA_raw_series（原始的 SLA_raw，pandas Series，索引是 lead_indices）
# - SLA_smoothed_raw（在 lead 上平滑后的值，Series，同样索引是 lead_indices）

plt.figure(figsize=(15, 6))

# Step 1: 原始 SLA_raw
plt.plot(SLA_raw_series.index, SLA_raw_series.values, 'o-', label='SLA_raw (lead only)', color='gray')

# Step 2: 平滑后 SLA
plt.plot(SLA_smoothed_raw.index, SLA_smoothed_raw.values, 'o-', label='SLA_smoothed_raw (lead only)', color='blue')

# Step 3: 插值得到的连续 SLA（整条轨迹）
plt.plot(df_L1.index, df_L1['SLA_interp'], '-', label='SLA_interp (interpolated full track)', color='red')

# Step 4: 最终平滑后的 SLA
plt.plot(df_L1.index, df_L1['SLA'], '-', label='SLA (final smoothed)', color='green')

plt.xlabel('Index (Along-track point)')
plt.ylabel('Sea Level Anomaly (m)')
plt.title('SLA Calculation Steps: Raw → Smoothed → Interpolated')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# 原始数据 df 的分布
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for t, color in [('lead', 'blue'), ('ice', 'red')]:
    mask = df['type'] == t
    plt.scatter(df.loc[mask, 'lon'], df.loc[mask, 'lat'], label=t, s=5, alpha=0.5, c=color)
plt.title("Original df['type'] distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(True)

# 筛选后 df_L1 的分布
plt.subplot(1, 2, 2)
for t, color in [('lead', 'blue'), ('ice', 'red')]:
    mask = df_L1['type'] == t
    plt.scatter(df_L1.loc[mask, 'lon'], df_L1.loc[mask, 'lat'], label=t, s=5, alpha=0.5, c=color)
plt.title("df_L1['type'] distribution")
plt.xlabel("Longitude")
plt.ylabel("Latitude")
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()


In [ ]:
# Plot waveforms for 'lead' and 'ice' categories
plt.figure(figsize=(12, 6))

# Plot waveforms for 'lead' category
plt.subplot(1, 2, 1)
lead_mask = df['type'] == 'lead'
if lead_mask.any():
    for waveform in waveform_20_ku[lead_mask][:50]:  # Only plot the first 50 waveforms
        plt.plot(waveform, color='blue', alpha=0.1)
    plt.plot(waveform_20_ku[lead_mask][:50].mean(axis=0), label='lead average waveform', color='blue', linewidth=2)
plt.xlabel('Range Bin')
plt.ylabel('Amplitude')
plt.title('Waveform Distribution for Lead')
plt.legend()
plt.grid(alpha=0.3)

# Plot waveforms for 'ice' category
plt.subplot(1, 2, 2)
ice_mask = df['type'] == 'ice'
if ice_mask.any():
    for waveform in waveform_20_ku[ice_mask][:50]:  # Only plot the first 50 waveforms
        plt.plot(waveform, color='red', alpha=0.1)
    plt.plot(waveform_20_ku[ice_mask][:50].mean(axis=0), label='ice average waveform', color='red', linewidth=2)
plt.xlabel('Range Bin')
plt.ylabel('Amplitude')
plt.title('Waveform Distribution for Ice')
plt.legend()
plt.grid(alpha=0.3)

plt.xlabel('Range Bin')
plt.ylabel('Amplitude')
plt.title('Waveform Distribution for Lead and Ice Categories')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

print(noise_power_20_ku[100])
print(noise_power_20_ku[1000])